# A2.4 · The NHI governance gap

**Function A — Security Architecture & Platform → The Identity & Non-Human Identity Engineer**  ·  *Security of AI*

---

**Risk.** Revoking one misbehaving agent breaks forty others.

**Control.** Enrolment, ownership, scope, expiry, attribution — an agent registry with honest enforcement limits.

**This lab.** Revoke exactly one agent without collateral.

| | |
|---|---|
| Open-source tooling | Keycloak, SPIRE |
| Open-weight models | — |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("A2.4"))

Non-human identities outnumber humans by an order of magnitude and are governed by roughly none of the same process. The gap is not policy — it is that nobody can enumerate them.

In [ ]:
from cybercommons import identity

reg = identity.Registry()
root = reg.record(identity.mint("alice"))
for actor, scopes in [("reviewer-agent", {"repo:read"}),
                      ("patch-agent",    {"repo:read", "repo:write"}),
                      ("deploy-agent",   {"repo:read"})]:
    reg.record(identity.exchange(root, actor, scopes))

print(f"{'identity':18s}{'tokens':>7}  scopes")
for row in reg.inventory():
    print(f"{row['actor']:18s}{row['tokens']:>7}  {row['scopes']}")

Now the governance question that separates an identity from a password: can you revoke exactly one of these?

In [ ]:
n = reg.revoke("patch-agent")
print(f"revoked patch-agent → {n} token(s) invalidated")
for row in reg.inventory():
    print(f"  {row['actor']:18s} revoked={row['revoked']}")
print("\nThe others keep working. If your only lever were rotating a shared")
print("secret, revoking one agent would take down all four.")

### Expect

Four identities are listed with their scopes. Revoking `patch-agent` marks only that row revoked; the other three remain usable.

### Your turn

Count the non-human identities in one production account. Then count how many have a named owner and an expiry. The ratio is your NHI governance gap.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/A2.4.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*